# box-array-to-tensor-with-recipe — ex3: build_parents: argnum → MiniTensor dict, filtered by requires_grad

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `box-array-to-tensor-with-recipe`. Running the final beacon cell reports progress against the `Backprop: Box array as Tensor + recipe` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: Box array as Tensor + recipe` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`box-array-to-tensor-with-recipe`** (exercise 3). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "box-array-to-tensor-with-recipe"
DD_SUBTOPIC = "Backprop: Box array as Tensor + recipe"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `build_parents` — the bookkeeping step that lets backward find the inputs

Ex1 boxed the raw output + recipe. Ex2 composed the whole wrapper. The third facet of the same atom is the parents-dict construction — the bookkeeping that lets the reverse pass know which input occupied which argument slot:

```python
parents = {
    idx: a for idx, a in enumerate(args)
    if isinstance(a, MiniTensor) and a.requires_grad
}
```

**Why a dict keyed by argnum.** Each back fn for `op(x, y)` needs to know which gradient goes to x and which goes to y — they're routed by ARGUMENT INDEX. `parents[0]` is the first MiniTensor input, `parents[1]` the second, etc. Skipping non-MiniTensor args means indices in `parents` are NOT contiguous: `op(t1, 3.0, t2)` produces `parents = {0: t1, 2: t2}`.

**Why filter on `requires_grad`.** A MiniTensor with `requires_grad=False` is a frozen input — gradient flow stops there. Including it in `parents` would waste a back-fn dispatch and corrupt the leaf-set the reverse pass uses to know when to stop. Filtering at build time keeps the graph minimal.

**Why skip when `requires_grad` is False.** No-grad inputs don't need to be revisited on backward. Filtering keeps the parents dict small and the reverse pass fast — especially important when one of the inputs is a giant constant (e.g. positional embeddings) that the user explicitly froze.

### Exercise 3 — build_parents: argnum → MiniTensor dict, filtered by requires_grad

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the parents-dict construction step of wrap_forward_fn: walk positional args once and yield a dict keyed by argument index, containing only the MiniTensors whose requires_grad is True.
> Keywords: parents, argnum, build, requires-grad, bookkeeping
> ```

**KCs targeted:** `parents-dict-by-argidx`, `filter-on-requires-grad`

Implement `ex3_build_parents(args)`. The wrapper has already completed unbox/coerce; this is the bookkeeping pass that records WHICH input occupied WHICH argument slot, so the reverse pass can route gradients back by argnum.

Inputs:
- `args`: tuple of positional inputs (mixed `MiniTensor`, raw `torch.Tensor`, Python scalars, etc.).

Output: a `dict[int, MiniTensor]` where:

- Keys are the original positional indices (0-based) — NOT renumbered.
- Values are the MiniTensor instances themselves (identity, not copies).
- Include an arg ONLY when `isinstance(a, MiniTensor)` AND `a.requires_grad` is True.
- Non-MiniTensor args and rg=False MiniTensors are SKIPPED — their index does NOT appear in the dict.

Examples:

```
build_parents((t_rg, 3.0, t_rg2))   → {0: t_rg, 2: t_rg2}
build_parents((t_no, t_rg))         → {1: t_rg}
build_parents(())                   → {}
build_parents((3.0, 4.0))           → {}
```

Constraints:
- Indices in the output are NOT contiguous when scalars / rg=False inputs intersperse — that's by design.
- Values must be the same Python object (not a copy).

In [ ]:
def ex3_build_parents(args):
    return {
        idx: a for idx, a in enumerate(args)
        if isinstance(a, MiniTensor) and a.requires_grad
    }


<details><summary>Solution</summary>

```python
def ex3_build_parents(args):
    return {
        idx: a for idx, a in enumerate(args)
        if isinstance(a, MiniTensor) and a.requires_grad
    }
```

**Indices are ORIGINAL, not renumbered.** `build_parents((m, 3.0, m2))` must return `{0: m, 2: m2}`, NOT `{0: m, 1: m2}`. The argnum is what back fns dispatch on — `multiply_back0(grad, ...)` knows it's working on argument 0, `multiply_back1` on argument 1. Renumbering would send gradients to the wrong slots.

**Dict comprehension over `enumerate(args)` is canonical.** Single-pass, O(n), no temp lists, indices and elements paired up naturally. The two-condition filter (`isinstance` AND `requires_grad`) is short-circuit — `requires_grad` is only accessed if the type check passed, so we never `AttributeError` on a non-MiniTensor.

**Why this is a third distinct facet.** Ex1 boxed the output + recipe. Ex2 composed the wrapper. Ex3 is the bookkeeping primitive that ex2 calls internally — same scan as `unbox_args`, different filter and different output shape. The wrapper actually combines all three: unbox, build_parents, then box-with-recipe.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex3',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()